# TP Jour 4 — Inférence Statistique, Segmentation Clients RFM & Restitution BI

### Objectifs de la séance
1. Mener des **tests statistiques non-paramétriques** (`scipy.stats` : **Mann-Whitney U** sur les délais régionaux et corrélation de **Spearman** entre retard et note d'avis).
2. Construire une **segmentation clients RFM** (*Recency, Frequency, Monetary*) à la maille **`customer_unique_id`** avec transformation `np.log1p`, `StandardScaler` et clustering **K-Means**, puis l'écrire dans **`/lakehouse/gold/customer_rfm`**.
3. Enrichir l'application **`app/dashboard.py`** pour restituer vos tables Gold sur `http://localhost:8501`.

> **Auto-validation :** Lancez `task test:j4` (ou `task test:all` pour valider les 4 jours d'un coup !).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_RFM_Stats_Dashboard")
    .getOrCreate()
)

## 1. Tests Statistiques Non-Paramétriques (`scipy.stats`)

### Exercice 1.1 — Test de Mann-Whitney U (Sud-Est vs Nord-Est) & Corrélation de Spearman
1. Extrayez depuis `/lakehouse/silver/orders` (commandes `delivered` avec `delivery_days` non nul) uniquement les colonnes `customer_state`, `delivery_days`, `delay_days`, `review_score` vers Pandas (`.toPandas()`).
2. Comparez les délais de livraison entre le **Sud-Est** (`SP`, `RJ`, `MG`) et le **Nord-Est** (`BA`, `PE`, `CE`) à l'aide de `stats.mannwhitneyu(...)`. Pourquoi un test *t* de Student n'est-il pas approprié ici ?
3. Calculez la corrélation de rangs de Spearman (`stats.spearmanr(...)`) entre `delay_days` et `review_score`.

In [ ]:
# TODO : Extraire l'échantillon projeté depuis Silver et exécuter Mann-Whitney U + Spearman
df_stats = None
assert df_stats is not None, "TODO : Extrayez `df_stats` depuis `/lakehouse/silver/orders`."

u_stat, p_val_mw = ...
rho, p_val_sp = ...

print(f"Mann-Whitney U = {u_stat:,.0f} (p = {p_val_mw:.3e})")
print(f"Spearman rho   = {rho:.4f} (p = {p_val_sp:.3e})")

## 2. Pipeline de Segmentation RFM + K-Means (`/lakehouse/gold/customer_rfm`)

> **Point d'attention schéma Olist :** Utilisez impérativement **`customer_unique_id`** (identifiant réel du client) et non `customer_id` (généré à chaque commande) !

### Exercice 2.1 — Calculer les indicateurs RFM en PySpark et segmenter avec K-Means
1. Calculez la date maximale d'achat `ref_ts = max(order_purchase_timestamp)` sur les commandes livrées.
2. Groupez `/lakehouse/silver/orders` par `customer_unique_id` pour calculer :
   - `recency_days` : `F.datediff(F.lit(ref_ts), F.max("order_purchase_timestamp"))`
   - `frequency` : `F.countDistinct("order_id")`
   - `monetary` : `F.round(F.sum("total_amount"), 2)` (filtrez `monetary > 0`)
   - `avg_review_score` et `customer_state` (`F.first(..., ignorenulls=True)`)
3. Récupérez le DataFrame agrégé avec `.toPandas()`, appliquez `np.log1p(...)` puis `StandardScaler()` sur `[recency_days, frequency, monetary]`, et entraînez `KMeans(n_clusters=4, random_state=42, n_init=10)`.
4. Associez un nom métier lisible dans une colonne `segment` (ex: `"VIP / Grands Paniers"`, `"Nouveaux / Récents"`, `"Fidèles (Multi-achats)"`, `"En sommeil / À réactiver"`) et sauvegardez le résultat au format **Delta** dans **`/lakehouse/gold/customer_rfm`**.

In [ ]:
# TODO : Calculer la table RFM par `customer_unique_id`, appliquer log1p + StandardScaler + KMeans(4),
# ajouter la colonne `segment` et écrire dans `/lakehouse/gold/customer_rfm`
rfm_pd = None
assert rfm_pd is not None, "TODO : Construisez le DataFrame `rfm_pd` et sauvegardez `/lakehouse/gold/customer_rfm`."

...

rfm_check = spark.read.format("delta").load("/lakehouse/gold/customer_rfm")
for col_req in ["customer_unique_id", "recency_days", "frequency", "monetary", "segment"]:
    assert col_req in rfm_check.columns, f"Colonne requise manquante dans customer_rfm : {col_req}"
print(f"Table /lakehouse/gold/customer_rfm validée ({rfm_check.count():,} clients segmentés) !")

## 3. Restitution Finale dans `app/dashboard.py` (Streamlit)

Ouvrez maintenant le fichier **`app/dashboard.py`** dans VS Code (ou JupyterLab) :
1. Chargez vos tables `/lakehouse/gold/kpi_categories`, `/lakehouse/gold/kpi_states`, `/lakehouse/gold/kpi_monthly` et `/lakehouse/gold/customer_rfm` via `load_gold_table(...)`.
2. Créez vos graphiques interactifs **Plotly Express** (`px.bar`, `px.scatter`, `px.line`, `px.pie`).
3. Accompagnez chaque visualisation d'un `st.expander("Requête SQL & Justification statistique")` expliquant vos choix méthodologiques.
4. Vérifiez le rendu en direct sur **http://localhost:8501** !